# Exploring College Basketball Data

## API Call

In [22]:
import requests
import pandas as pd
from cbb_predictor.config import CBBD_API_KEY, CBBD_BASE_URL

response = requests.get(
    f"{CBBD_BASE_URL}/games",
    headers={"Authorization": f"Bearer {CBBD_API_KEY}"},
    params={"season": 2027},
    timeout=30,
)
response.raise_for_status()
games = pd.json_normalize(response.json())
games.shape

(3000, 39)

## Exploring

### How are seasons numbered

In [23]:
print(games["seasonLabel"].unique())
print(games["startDate"].min(), games["startDate"].max())

<ArrowStringArray>
['20262027']
Length: 1, dtype: str
2026-11-01T14:30:00.000Z 2027-01-02T19:00:00.000Z


### Note
This means seasons are labeled by ending year. So the 2026 - 2027 season is labeled with 2027.

### What are the collumns

In [24]:
print(games.columns.tolist())

['id', 'sourceId', 'seasonLabel', 'season', 'seasonType', 'tournament', 'startDate', 'startTimeTbd', 'neutralSite', 'conferenceGame', 'gameType', 'status', 'gameNotes', 'attendance', 'homeTeamId', 'homeTeam', 'homeConferenceId', 'homeConference', 'homeSeed', 'homePoints', 'homePeriodPoints', 'homeWinner', 'homeTeamEloStart', 'homeTeamEloEnd', 'awayTeamId', 'awayTeam', 'awayConferenceId', 'awayConference', 'awaySeed', 'awayPoints', 'awayPeriodPoints', 'awayWinner', 'awayTeamEloStart', 'awayTeamEloEnd', 'excitement', 'venueId', 'venue', 'city', 'state']


### What are the status values

In [25]:
games["status"].value_counts()

status
scheduled    3000
Name: count, dtype: int64

### Note
Currently the only status is scheduled, but there are also final, in_progress, postponed / canceled. Each will be handled differently.

### How many games and when does season start

### Note
It may appear that there are only 3000 games because that is how many rows are returned. However, the API call only returns the next 3000 games, which is about half of the actual total. This can be seen when finding the max startDate, which returns January 2nd, 2027.

In [26]:
games["startDate"].max()

'2027-01-02T19:00:00.000Z'

### Note
Instead, we can make seperate calls for each month and combining them.

In [27]:
months = [("2026-11-01T00:00:00Z", "2026-12-01T00:00:00Z"), ("2026-12-01T00:00:00Z", "2027-01-01T00:00:00Z"), ("2027-01-01T00:00:00Z", "2027-02-01T00:00:00Z"), ("2027-02-01T00:00:00Z", "2027-03-01T00:00:00Z"), ("2027-03-01T00:00:00Z", "2027-04-01T00:00:00Z"), ("2027-04-01T00:00:00Z", "2027-05-01T00:00:00Z")]
frames = []

for start, end in months:
    response = requests.get(
        f"{CBBD_BASE_URL}/games",
        headers={"Authorization": f"Bearer {CBBD_API_KEY}"},
        params={"season": 2027, "startDateRange": start, "endDateRange": end},
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    if len(data) >= 3000:
        raise ValueError(f"{start} to {end} hit the 3,000-game limit; split it into smaller ranges")
    frames.append(pd.json_normalize(response.json()))

games = pd.concat(frames, ignore_index=True).drop_duplicates(subset="id")
games.shape



(6006, 39)

In [28]:
print(len(games))
print(games["startDate"].min(), games["startDate"].max())
print(games.groupby(games["startDate"].str[:7]).size())

6006
2026-11-01T14:30:00.000Z 2027-03-08T00:00:00.000Z
startDate
2026-11    1588
2026-12    1313
2027-01    1570
2027-02    1331
2027-03     204
dtype: int64


### Notes
This shows there are a total of 6,006 games in the 2026-27 season This season starts on November, 1st and ends on March, 8th. It can be seen that this dataset doesn't show games for the end of the year tournaments. This is because those games aren't scheduled yet, so the pipeline must treat the schedule as a moving target and refetch the schedule data to catch when the schedule is updated.

### Time zone

In [29]:
games["startDate"].head()

0    2026-11-01T14:30:00.000Z
1    2026-11-02T04:59:00.000Z
2    2026-11-02T05:00:00.000Z
3    2026-11-02T05:00:00.000Z
4    2026-11-02T05:00:00.000Z
Name: startDate, dtype: str

In [30]:
games["StartET"] = pd.to_datetime(games["startDate"], utc=True).dt.tz_convert("America/New_York")
games["Date"] = games["StartET"].dt.tz_localize(None).dt.normalize()
games[["StartET", "startDate"]].head()

,StartET,startDate
0,2026-11-01 09:30:00-05:00,2026-11-01T14:30:00.000Z
1,2026-11-01 23:59:00-05:00,2026-11-02T04:59:00.000Z
2,2026-11-02 00:00:00-05:00,2026-11-02T05:00:00.000Z
3,2026-11-02 00:00:00-05:00,2026-11-02T05:00:00.000Z
4,2026-11-02 00:00:00-05:00,2026-11-02T05:00:00.000Z


### Note
startDate is in UTC (represented by the Z). Evening games will cross midnight so all games will be converted to Eastern Time before use.

## Key Takeaways

* This gives a away to pull the schedule daily
* Only 3000 games are pulled per call, so there needs to be multiple calls made to get the entire season
* The season is labeled by the year at the end of the season
* There are several status values that need to be handled differently
* The timezone is defaulted to UTC, a conversion to ET must be done before use